In [86]:
!pip install openai tqdm

In [1]:
import pandas as pd
import json
import numpy as np
from IPython.display import JSON


In [2]:
graphs = pd.read_json('train_sceneGraphs.json')
questions = pd.read_json('train_balanced_questions.json')

In [7]:
graphs = graphs.T
questions = questions.T

In [27]:
detailed_types = []
for i in questions.types:
    detailed_types.append(i['detailed'])

In [41]:
set(detailed_types)

{'activity',
 'activityChoose',
 'activityWho',
 'category',
 'categoryAttr',
 'categoryRelO',
 'categoryRelOChoose',
 'categoryRelS',
 'categoryThat',
 'categoryThatChoose',
 'categoryThis',
 'categoryThisChoose',
 'chooseAttr',
 'company',
 'companyChoose',
 'companyVerify',
 'companyVerifyC',
 'comparativeChoose',
 'diffAnimals',
 'diffAnimalsC',
 'diffGender',
 'dir',
 'directOf',
 'directWhich',
 'exist',
 'existAnd',
 'existAndC',
 'existAttr',
 'existAttrC',
 'existAttrNot',
 'existAttrNotC',
 'existAttrOr',
 'existAttrOrC',
 'existC',
 'existMaterial',
 'existMaterialC',
 'existMaterialNot',
 'existMaterialNotC',
 'existOr',
 'existOrC',
 'existRelS',
 'existRelSC',
 'existRelSRC',
 'existThat',
 'existThatC',
 'existThatNot',
 'existThatNotC',
 'existThatOr',
 'existThatOrC',
 'how',
 'locationChoose',
 'locationVerify',
 'locationVerifyC',
 'material',
 'materialChoose',
 'materialVerify',
 'materialVerifyC',
 'objThisChoose',
 'place',
 'placeChoose',
 'placeVerify',
 'place

In [42]:
relevant_question_types = ['chooseRel', 'verifyRel', 'existRel', 'chooseObjRel', 'compare', 'logicAnd', 'logicOr', 'twoSame', 'twoDiff', 'allSame', 'allDiff']
relevant_question_types_real = ['compare', 'chooseAttr', 'diffAnimals', 'diffAnimalsC', 'diffGender', 'existRelS', 'existRelSC', 'existRelSRC', 
                                'placeVerify', 'placeVerifyC', 'relVerify', 'relVerifyCop', 'relVerifyCr', 'twoDifferent', 'twoDifferentC', 'twoSame', 'twoSameC']

In [47]:
relevant_questions = questions[[q['detailed'] in relevant_question_types_real for q in questions.types]]

In [60]:
relevant_questions

,semantic,entailed,equivalent,question,imageId,isBalanced,groups,answer,semanticStr,annotations,types,fullAnswer
15736264,"[{'operation': 'select', 'dependencies': [], '...","[15736259, 15736258, 15736267, 15736253, 15736...",[15736264],Is the tall clock small or large?,2368326,True,"{'global': 'size', 'local': '10c-clock_size'}",large,select: clock (746851)->filter height: tall [0...,"{'answer': {}, 'question': {'2:4': '746851'}, ...","{'detailed': 'chooseAttr', 'semantic': 'attr',...",The clock is large.
7452748,"[{'operation': 'select', 'dependencies': [], '...",[],[07452748],Is the cheese to the left of the food on the p...,2325360,True,"{'global': None, 'local': '13-cheese_food'}",yes,"select: cheese (2879218)->verify rel: food,to ...","{'answer': {}, 'question': {'11': '3941619', '...","{'detailed': 'relVerify', 'semantic': 'rel', '...","Yes, the cheese is to the left of the food."
1855103,"[{'operation': 'select', 'dependencies': [], '...",[],[01855103],Is the bird wearing a coat?,2355018,True,"{'global': None, 'local': '13-bird_coat'}",yes,"select: bird (1901864)->verify rel: coat,weari...","{'answer': {}, 'question': {'2': '1901864', '5...","{'detailed': 'relVerify', 'semantic': 'rel', '...","Yes, the bird is wearing a coat."
18628483,"[{'operation': 'select', 'dependencies': [], '...","[18628423, 18628484, 18628485]","[18628483, 18628484]",Are there any people to the right of the mirro...,2356876,True,"{'global': None, 'local': '13-mirror_people'}",yes,select: mirror (2127390)->filter vposition: to...,"{'answer': {}, 'question': {'9': '2127390', '3...","{'detailed': 'existRelS', 'semantic': 'rel', '...","Yes, there are people to the right of the mirror."
367457,"[{'operation': 'select', 'dependencies': [], '...",[],[0367457],Is the skier to the right of a helmet?,713265,True,"{'global': None, 'local': '13-skier_helmet'}",yes,"select: skier (1581318)->verify rel: helmet,to...","{'answer': {}, 'question': {'8': '3700428', '2...","{'detailed': 'relVerify', 'semantic': 'rel', '...","Yes, the skier is to the right of a helmet."
...,...,...,...,...,...,...,...,...,...,...,...,...
3279256,"[{'operation': 'select', 'dependencies': [], '...",[03279255],[03279256],Is the white vehicle in front of the motorcycl...,2385536,True,"{'global': None, 'local': '13-car_motorcycle'}",yes,select: vehicle (521218)->filter color: white ...,"{'answer': {}, 'question': {'11': '521216', '8...","{'detailed': 'relVerify', 'semantic': 'rel', '...","Yes, the car is in front of the motorbike."
7830238,"[{'operation': 'select', 'dependencies': [], '...","[07829951, 07829950]",[07830238],Is the silver tower to the left of the lamp th...,2382377,True,"{'global': None, 'local': '13-tower_lamp'}",yes,select: tower (1329072)->filter color: silver ...,"{'answer': {}, 'question': {'9': '1329063', '1...","{'detailed': 'relVerify', 'semantic': 'rel', '...","Yes, the tower is to the left of the lamp."
1861798,"[{'operation': 'select', 'dependencies': [], '...",[],[01861798],Are there any boats in the ocean?,2398967,True,"{'global': None, 'local': '13-ocean_boat'}",no,"select: ocean (4311159)->relate: boat,in,s (-)...","{'answer': {}, 'question': {'6': '4311159'}, '...","{'detailed': 'existRelSC', 'semantic': 'rel', ...","No, there is a man in the ocean."
4418580,"[{'operation': 'select', 'dependencies': [], '...",[04418581],[04418580],Is the man wearing shorts?,2384902,True,"{'global': None, 'local': '13-man_shorts'}",yes,"select: man (1303425)->verify rel: shorts,wear...","{'answer': {}, 'question': {'2': '1303425', '4...","{'detailed': 'relVerify', 'semantic': 'rel', '...","Yes, the man is wearing shorts."


In [69]:
relevant_questions.imageId.unique().astype(int)

array([2368326, 2325360, 2355018, ..., 2380058, 2337473, 2398967],
      shape=(49503,))

In [70]:
graphs_subset = graphs.loc[relevant_questions.imageId.unique().astype(int)]

In [72]:
relevant_questions.to_csv('relevant_questions.csv', index=False)
graphs_subset.to_csv('graphs_subset.csv', index=False)


## Seleciona do conjunto de validação

In [3]:
graphs = pd.read_json('data/val_sceneGraphs.json')
questions = pd.read_json('data/val_balanced_questions.json')

In [4]:
graphs = graphs.T
questions = questions.T

In [ ]:
relevant_question_types_real = ['compare', 'chooseAttr', 'diffAnimals', 'diffAnimalsC', 'diffGender', 'existRelS', 'existRelSC', 'existRelSRC', 
                                'placeVerify', 'placeVerifyC', 'relVerify', 'relVerifyCop', 'relVerifyCr', 'twoDifferent', 'twoDifferentC', 'twoSame', 'twoSameC']

In [6]:
relevant_questions = questions[[q['detailed'] in relevant_question_types_real for q in questions.types]]

In [ ]:
# Seleciona as 1000 questões com o maior comprimento da string semanticStr, que são as questões mais longas e complexas
relevant_questions = relevant_questions.assign(semanticStr_len=relevant_questions['semanticStr'].str.len()).sort_values(by="semanticStr_len").tail(1000).drop(columns="semanticStr_len")

In [21]:
graphs_subset = graphs.loc[relevant_questions.imageId.unique().astype(int)]

In [22]:
relevant_questions.to_csv('data/val_relevant_questions.csv', index=False)
graphs_subset.to_csv('data/val_graphs_subset.csv', index=False)